# Lesson 4 | How can a circuit remember the previous membrane potential?

The first three lessons still lived entirely in software: a Python variable `v` naturally retained the previous membrane-state value.

But our destination is a **Field-Programmable Gate Array (FPGA)**. Inside an FPGA there are no Python variables; there are digital circuits.

So today we solve one question only:

> **How does a digital circuit store state and update that state at the right time?**

The primary new concept is **digital state and the clock**.

You are still not expected to write SystemVerilog today.

## 1. What do we mean by a digital circuit?

A **digital circuit** represents and processes information using discrete states. The most basic unit is a **binary digit (bit)**, usually `0` or `1`.

In the previous lesson we saw that multiple bits can represent integers and fixed-point values.

Today we stay one level above transistor physics and focus on a simpler distinction:

- some circuits compute only from the inputs that exist right now;
- other circuits must also remember what happened earlier.

A neuron clearly needs the second kind because the next membrane state depends on the previous membrane state.

## 2. Computation without memory: combinational logic

**Combinational logic** produces outputs determined only by current inputs.

A simple example is addition: given `a` and `b`, produce `a+b`. If the inputs change, the output changes. The circuit itself does not remember yesterday's or the previous cycle's result.

The Python function below mimics that current-input-only relationship.

In [ ]:
def combinational_adder(a, b):
    return a + b

for a, b in [(1, 2), (4, 5), (10, -3)]:
    print(a, '+', b, '=', combinational_adder(a, b))

## 3. Why is combinational logic not enough for LIF?

If a LIF neuron sees only the current input `I[t]` and has no memory of the previous `V[t]`, it cannot integrate across time.

We therefore need hardware that holds a value so the next update can use it.

A small hardware structure that stores digital state is called a **register**.

For now, think of a register simply as:

> **a place in a circuit that can hold a digital state value.**

Real registers are built from lower-level storage elements, but transistor and flip-flop internals are not today's topic.

## 4. When does state update? Meet the clock

If many registers changed at arbitrary times, coordinating a large digital system would be difficult.

Synchronous digital systems commonly use a periodic signal that establishes when stored state may update. That signal is the **clock**.

The instant when the clock changes from low to high or high to low is a **clock edge**. Many registers are designed to accept a new value only on a chosen edge, such as the rising edge.

Build this intuition:

> **Between edges, circuitry computes a next state. At the clock edge, a register captures that next state and it becomes the new current state.**

## 5. Simulate “update only on the clock edge” in Python

Python is not hardware, but we can use it to model the rule first.

Treat each loop iteration below as if a clock edge occurred. `before` is the old state stored before the edge; `after_clock` is the new state stored after it.

In [ ]:
state = 0
next_values = [3, 7, 2, 9]

for cycle, next_value in enumerate(next_values):
    before = state

    # Imagine a clock edge here:
    state = next_value

    print(f'cycle={cycle}: before={before}, after_clock={state}')

## 6. `state` and `next_state` are different ideas

This is one of the most important hardware intuitions to build.

Suppose the register currently stores:

`state = 3`

Combinational logic computes:

`next_state = 5`

Before the clock edge, the register still stores `3`. The value `5` is only what we intend to store at the next edge.

At the edge:

`state ← next_state`

Only then does the stored state become `5`.

This should remind you of `candidate_v` versus `stored_v` in the earlier LIF lesson.

## 7. A minimal accumulator already looks neuron-like

An **accumulator** repeatedly adds new input into stored state. That matches the integrate intuition in LIF.

We add one supporting component: a **comparator**, which answers a question such as `next_state >= threshold ?`.

Comparator is supporting vocabulary, not the main concept today. For now, it simply compares two values.

In [ ]:
state = 0
threshold = 4
inputs = [1, 1, 1, 1, 2, 2]

for cycle, x in enumerate(inputs):
    before = state

    # Combinational calculation between clock edges
    next_state = state + x
    spike = next_state >= threshold
    value_to_store = 0 if spike else next_state

    # Imagine the clock edge here
    state = value_to_store

    print(
        f'cycle={cycle}: state_before={before}, input={x}, '
        f'next_state={next_state}, spike={spike}, state_after={state}'
    )

## 8. Read the system again as hardware roles

Ignore Python syntax and identify roles:

1. `state`: the current value stored by a register;
2. `state + x`: combinational logic that computes a next state;
3. `next_state >= threshold`: a comparator;
4. `value_to_store`: the value chosen for the next register update;
5. clock edge: the moment the register captures that value.

A rough conceptual picture is:

```text
current state ----> calculation ----> next state candidate
     ^                                     |
     |                                     v
  register <---- value chosen for storage ----
     ^
     |
 clock edge decides when storage updates
```

That is already close to the skeleton of a digital LIF neuron.

## 9. Combinational versus sequential logic

We can now introduce a second term:

**Sequential logic** is digital logic whose behavior depends on previously stored state as well as current input.

A rough comparison:

| Type | Remembers past state? | Possible role in our neuron |
|---|---|---|
| combinational logic | no | addition, leak arithmetic, threshold comparison |
| sequential logic | yes | stored membrane state, refractory counter |

Do not interpret `sequential` as “execute software lines one by one.” Here it means the system state evolves across a sequence of clock cycles.

## 10. Try It: draw a timing table before running

Given:

- initial `state = 0`
- inputs = `[2, 1, 3]`
- threshold = `5`

Do not run first. Fill in:

| cycle | state before | input | next state | spike? | state after clock |
|---|---:|---:|---:|---|---:|
| 0 | ? | 2 | ? | ? | ? |
| 1 | ? | 1 | ? | ? | ? |
| 2 | ? | 3 | ? | ? | ? |

Then modify the code to verify your table.

If your prediction differs, first inspect your understanding of state/next state rather than immediately asking AI to rewrite the code.

## 11. What are we deliberately not learning today?

These terms will matter later, but they are previews only:

- **Hardware Description Language (HDL)**: a class of languages used to describe digital hardware;
- **SystemVerilog**: the hardware design and verification language planned for this project;
- **Register-Transfer Level (RTL)**: a design level describing what registers store and how data is computed/transferred across clock cycles.

You do not need to write HDL or RTL today. Today's goal is only state, next state, register, and clock edge.

## 12. AI Task

Ask AI to draw an ASCII diagram of:

`register → calculation → next_state → register`

and answer:

- which parts are combinational logic?
- which part stores state?
- where does the clock edge matter?

Do not ask it to generate a full LIF SystemVerilog module yet. We have not learned enough to review one responsibly.

## 13. Human Check

Without AI, explain:

- why combinational logic alone cannot implement LIF;
- what a register is;
- the difference between clock and clock edge;
- why `state` and `next_state` must not be confused;
- why `sequential logic` does not mean “software lines execute sequentially”;
- why we intentionally have not started SystemVerilog yet.

## 14. Engineering Handoff

The next stage turns today's intuition into three real digital-hardware micro-experiments:

1. combinational adder;
2. clocked counter;
3. accumulator + threshold.

Only after those will we implement the formal LIF RTL.

This follows our Learning Independence Axiom: the first day of learning a clock should not also require mastering SystemVerilog syntax and a complete neuron implementation.

## 15. Project Trace

- Lesson ID: `LSN-004`
- Engineering slice: `RMD-003A`
- Learning-process requirement/design: `PFR1 / PDP1`
- Prepares for product design: `FR2 / DP2`

These IDs maintain the project; learners do not need to memorize them.

## 16. Exit Ticket

Before continuing, you should be able to:

1. expand and explain FPGA, bit, register, clock, and clock edge;
2. explain combinational versus sequential logic;
3. manually write three or four cycles of `state` and `next_state` for a simple accumulator;
4. clearly state that today built digital-state intuition, not mastery of HDL / RTL / SystemVerilog.

The next lesson takes another small step:

> Besides remembering state, what basic logical building blocks does a digital circuit need?